******Car-Dheko_Used_Car_Price_Prediction******

There are 6 files of cities data

****Data Processing****

**a)	Import and concatenate:**

i)	Import all city’s dataset which is in unstructured format.

ii)	Convert it into a  structured format.

iii)Added a new column named ‘City’ and assign values for all rows with the name of the respective city.

iv)	Concatenate all datasets and make it as a single dataset.

**Banglore**

Perfectly Done unstructured to structured Banglore cars

In [30]:
import os
import pandas as pd
import ast
from collections import defaultdict

input_folder = "csv_files"
output_file = os.path.join(input_folder, "structured_cars_data.csv")

city_map = {
    "bangalore_cars.csv": "BANGALORE",
    "chennai_cars.csv": "CHENNAI",
    "delhi_cars.csv": "DELHI",
    "hyderabad_cars.csv": "HYDERABAD",
    "jaipur_cars.csv": "JAIPUR",
    "kolkata_cars.csv": "KOLKATA"
}

def parse_dict_column(column):
    return column.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)

def flatten_overview(overview_data):
    if isinstance(overview_data, dict):
        return {item['key']: item['value'] for item in overview_data.get("top", []) if isinstance(item, dict)}
    return {}

def flatten_feature(data):
    features = []
    if isinstance(data, dict):
        top = data.get("top", [])
        features.extend(f.get("value") for f in top if isinstance(f, dict))
        for section in data.get("data", []):
            if isinstance(section, dict):
                for item in section.get("list", []):
                    if isinstance(item, dict):
                        val = item.get("value")
                        if val:
                            features.append(val)
    return {'Feature_' + str(i): v for i, v in enumerate(features)}

def flatten_specs(data):
    specs = {}
    if isinstance(data, dict):
        for item in data.get("top", []):
            if isinstance(item, dict):
                k = item.get("key")
                v = item.get("value")
                if k and v:
                    specs[k] = v
        for section in data.get("data", []):
            if isinstance(section, dict):
                for item in section.get("list", []):
                    if isinstance(item, dict):
                        k = item.get("key")
                        v = item.get("value")
                        if k and v:
                            specs[k] = v
    return specs

def deduplicate_columns(columns):
    seen = defaultdict(int)
    new_columns = []
    for col in columns:
        if seen[col] == 0:
            new_columns.append(col)
        else:
            new_columns.append(f"{col}.{seen[col]}")
        seen[col] += 1
    return new_columns

dfs_combined = []

for file_name in os.listdir(input_folder):
    if file_name in city_map:
        file_path = os.path.join(input_folder, file_name)
        print(f"📥 Processing: {file_path}")
        
        try:
            df = pd.read_csv(file_path)

            for col in ["new_car_detail", "new_car_overview", "new_car_feature", "new_car_specs"]:
                if col in df.columns:
                    df[col] = parse_dict_column(df[col])

            df_overview = pd.DataFrame(df["new_car_overview"].apply(flatten_overview).tolist()) if "new_car_overview" in df.columns else pd.DataFrame()
            df_detail = pd.json_normalize(df["new_car_detail"]) if "new_car_detail" in df.columns else pd.DataFrame()
            df_feature = pd.DataFrame(df["new_car_feature"].apply(flatten_feature).tolist()) if "new_car_feature" in df.columns else pd.DataFrame()
            df_specs = pd.DataFrame(df["new_car_specs"].apply(flatten_specs).tolist()) if "new_car_specs" in df.columns else pd.DataFrame()
            base_cols = df[["car_links"]] if "car_links" in df.columns else pd.DataFrame()

            df_final = pd.concat([base_cols, df_detail, df_overview, df_feature, df_specs], axis=1)
            df_final.columns = deduplicate_columns(df_final.columns)

            df_final["City"] = city_map[file_name]
            dfs_combined.append(df_final)

        except Exception as e:
            print(f"❌ Error processing {file_name}: {e}")

# Save final output
if dfs_combined:
    final_df = pd.concat(dfs_combined, ignore_index=True)
    final_df.to_csv(output_file, index=False)
    print(f"✅ Final structured data saved to: {output_file}")
else:
    print("⚠️ No data was processed.")


📥 Processing: csv_files\bangalore_cars.csv
📥 Processing: csv_files\chennai_cars.csv
📥 Processing: csv_files\delhi_cars.csv
📥 Processing: csv_files\hyderabad_cars.csv
📥 Processing: csv_files\jaipur_cars.csv
📥 Processing: csv_files\kolkata_cars.csv
✅ Final structured data saved to: csv_files\structured_cars_data.csv


In [7]:
import pandas as pd

file_name=r"C:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\csv_files\structured_cars_data.csv"

# Read the CSV file into a Pandas DataFrame
df_check = pd.read_csv(file_name)

nan_summary = pd.DataFrame({
    'Column': df_check.columns,
    'Data Type': df_check.dtypes,
    'Non-Null Count': df_check.notna().sum(),
    'NaN Count': df_check.isna().sum(),
    'NaN Percentage': (df_check.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

C:\Users\USER\AppData\Local\Temp\ipykernel_23328\1927730870.py:6: DtypeWarning: Columns (14,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,208,209,210,211,212,213,214,215,216,217,218) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(file_name)


                       Column Data Type  Non-Null Count  NaN Count  NaN Percentage
0                   car_links    object            8369          0            0.00
1                          it     int64            8369          0            0.00
2                          ft    object            8369          0            0.00
3                          bt    object            8365          4            0.05
4                          km    object            8369          0            0.00
5                transmission    object            8369          0            0.00
6                     ownerNo     int64            8369          0            0.00
7                       owner    object            8369          0            0.00
8                         oem    object            8369          0            0.00
9                       model    object            8369          0            0.00
10                  modelYear     int64            8369          0            0.00
11  

In [11]:
df_check[['price','bt', 'Kms Driven','owner','Year of Manufacture','Mileage','Fuel Type','Registration Year','modelYear', 'Insurance Validity','Gear Box', 'modelYear', 'Transmission', 'Seats', 'City', 'Engine Displacement']].tail(10)

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,modelYear,Transmission,Seats,City,Engine Displacement
8359,₹ 2.50 Lakh,Sedan,"60,000 Kms",1st Owner,2010.0,17.5 kmpl,Petrol,2010,2010,Third Party insurance,5 Speed,2010,Manual,5 Seats,KOLKATA,1298 cc
8360,₹ 1 Lakh,Hatchback,"30,000 Kms",3rd Owner,2002.0,16.1 kmpl,Petrol,2002,2002,Third Party insurance,4 Speed,2002,Manual,4 Seats,KOLKATA,796 cc
8361,₹ 4 Lakh,Sedan,"60,000 Kms",1st Owner,2013.0,16.8 kmpl,Petrol,2013,2013,Third Party insurance,5 Speed,2013,Manual,5 Seats,KOLKATA,1497 cc
8362,₹ 3.60 Lakh,Hatchback,"70,000 Kms",1st Owner,2015.0,16.2 kmpl,Petrol,2015,2015,Third Party insurance,5 Speed,2015,Manual,5 Seats,KOLKATA,1199 cc
8363,₹ 1.50 Lakh,Hatchback,"30,000 Kms",1st Owner,2009.0,NaN,Petrol,2009,2009,Third Party insurance,5 Speed,2009,Manual,5 Seats,KOLKATA,1086 cc
8364,₹ 5.10 Lakh,Hatchback,"10,000 Kms",1st Owner,2022.0,25.24 kmpl,Petrol,2022,2022,Third Party insurance,5-Speed,2022,Manual,5 Seats,KOLKATA,998 cc
8365,₹ 1.80 Lakh,Hatchback,"1,20,000 Kms",1st Owner,2014.0,22.74 kmpl,Petrol,2014,2014,Third Party insurance,5 Speed,2014,Manual,5 Seats,KOLKATA,796 cc
8366,₹ 5.50 Lakh,Sedan,"50,000 Kms",3rd Owner,2011.0,11.74 kmpl,Petrol,2011,2011,Third Party insurance,7 Speed,2011,Automatic,5 Seats,KOLKATA,1796 cc
8367,₹ 1.40 Lakh,Hatchback,"40,000 Kms",1st Owner,2012.0,18.5 kmpl,Petrol,2012,2012,Third Party insurance,5 Speed,2012,Manual,5 Seats,KOLKATA,1197 cc
8368,₹ 5 Lakh,SUV,"1,20,000 Kms",2nd Owner,2017.0,19.72 kmpl,Diesel,2017,2017,Third Party insurance,6 Speed,2017,Manual,5 Seats,KOLKATA,1461 cc


✅ Selected Features and Justifications

| Column               | Description                                   | Justification                                                                 |
|----------------------|-----------------------------------------------|-------------------------------------------------------------------------------|
| `price`              | Selling price of the used car (Target variable) | This is the variable we're predicting.                                        |
| `bt`                 | Possibly body type or build type              | Vehicle type (e.g., SUV, sedan) affects demand, pricing, and buyer preference.|
| `Kms Driven`         | Total kilometers driven                        | Indicates vehicle usage; higher values usually reduce resale value.           |
| `owner`              | Number or type of previous owners              | Helps assess usage history and trust; fewer owners often mean better value.   |
| `Year of Manufacture`| Production year of the car                     | Indicates age; newer cars typically sell for higher prices.                   |
| `Mileage`            | Fuel efficiency (e.g., km/l)                   | Higher mileage is appealing and adds value to the car.                        |
|                         |
| `Fuel Type`          | Petrol, Diesel, CNG, Electric, etc.            | Different fuels affect running costs and buyer demand.                        |
| `Registration Year`  | Year the car was registered                    | Might differ from manufacturing year; important for insurance and resale.     |
| `modelYear`          | Year of manufacture                            | Reflects the age of the vehicle; newer cars tend to sell for higher prices.   |
| `Insurance Validity` | Remaining insurance period                     | A valid insurance policy adds value and trust for the buyer.                  |
| `Gear Box`           | Number of gears                                | Indicates car performance and class; more gears can mean better performance.  |
| `Transmission`       | Manual or Automatic                            | Automatics generally command a higher resale price, especially in cities.     |
| `Seats`              | Number of seats                                | More seating capacity appeals to families and commercial buyers.              |
| `City`               | City where the car is listed                   | Price trends vary by location due to demand, taxes, and road conditions.      |
| `Engine Displacement`| Engine size in CC                              | Affects performance, tax class, and buyer interest.                           |


In [12]:
df_check['Engine Displacement'].unique()

array(['998 cc', '1497 cc', '1199 cc', '1197 cc', '1248 cc', '1956 cc',
       '1198 cc', '1462 cc', '2179 cc', '1950 cc', '1396 cc', '1995 cc',
       '1498 cc', '4663 cc', '1086 cc', '1991 cc', '1968 cc', '1998 cc',
       '2982 cc', '1461 cc', '1797 cc', '796 cc', '1353 cc', '2925 cc',
       '2987 cc', '2967 cc', '999 cc', '1341 cc', '1496 cc', '1582 cc',
       '1798 cc', '1120 cc', '1969 cc', '1451 cc', '814 cc', '1999 cc',
       '1591 cc', '1397 cc', '1368 cc', '1598 cc', '3198 cc', '1196 cc',
       '2993 cc', '2755 cc', '2143 cc', '2354 cc', '1332 cc', '2199 cc',
       '1997 cc', '1499 cc', '2148 cc', '2696 cc', '1364 cc', '2494 cc',
       '1373 cc', '1299 cc', '1061 cc', '1493 cc', '2497 cc', '1495 cc',
       '1590 cc', '1896 cc', '1330 cc', '1781 cc', '1390 cc', '1194 cc',
       '2953 cc', '1399 cc', '1595 cc', nan, '2496 cc', '0 cc', '1984 cc',
       '2694 cc', '1298 cc', '2362 cc', '970 cc', '2198 cc', '1599 cc',
       '993 cc', '2393 cc', '1389 cc', '1597 cc', '249

Necessary  columns

In [2]:
import pandas as pd
import os

# Load the CSV file
file_path = "csv_files/structured_cars_data.csv"
df = pd.read_csv(file_path)

# Define the column name mappings
column_mapping = {
    'bt': 'BODY_TYPE',
    'Kms Driven': 'KILOMETERS_DRIVEN',
    'owner': 'NUMBER_OF_OWNERS',
    'Year of Manufacture': 'YEAR_OF_MANUFACTURE',
    'Fuel Type': 'FUEL_TYPE',
    'modelYear': 'MODEL_YEAR',
    'Insurance Validity': 'INSURANCE_VALIDITY',
    'Gear Box': 'NUMBER_OF_GEARS',
    'price': 'PRICE',
    'Mileage': 'MILEAGE',
    'Transmission': 'TRANSMISSION',
    'Seats': 'SEATS',
    'City': 'CITY_NAME'
}

# Create a dictionary for the final column names
new_columns = {}
for old_col in df.columns:
    if old_col in column_mapping:
        new_columns[old_col] = column_mapping[old_col]
    else:
        new_columns[old_col] = old_col  # Keep original case if not in mapping

# Rename the columns in the DataFrame
df_renamed = df.rename(columns=new_columns)

# Define wanted columns (using the *new* names from the mapping, otherwise original case)
wanted_columns_final = [
    'PRICE',
    'BODY_TYPE',
    'KILOMETERS_DRIVEN',
    'NUMBER_OF_OWNERS',
    'YEAR_OF_MANUFACTURE',
    'MILEAGE',
    'FUEL_TYPE',
    'MODEL_YEAR',
    'INSURANCE_VALIDITY',
    'NUMBER_OF_GEARS',
    'TRANSMISSION',
    'SEATS',
    'CITY_NAME'
]

# Filter only the columns that exist in the renamed DataFrame
wanted_columns_present = [col for col in wanted_columns_final if col in df_renamed.columns]

# Select and save the cleaned data
df_cleaned = df_renamed[wanted_columns_present]

# Make sure the output folder exists
os.makedirs("output", exist_ok=True)

# Save the DataFrame to CSV inside the output folder
df_cleaned.to_csv("csv_files/necessary_wanted_cols.csv", index=False)

print("✅ File saved to: output/necessary_wanted_cols.csv")

✅ File saved to: output/necessary_wanted_cols.csv


C:\Users\USER\AppData\Local\Temp\ipykernel_25204\336592845.py:6: DtypeWarning: Columns (14,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,208,209,210,211,212,213,214,215,216,217,218) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [3]:
nan_summary = pd.DataFrame({
    'Column': df_cleaned .columns,
    'Data Type': df_cleaned .dtypes,
    'Non-Null Count': df_cleaned.notna().sum(),
    'NaN Count': df_cleaned .isna().sum(),
    'NaN Percentage': (df_cleaned.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 PRICE    object            8369          0            0.00
1             BODY_TYPE    object            8365          4            0.05
2     KILOMETERS_DRIVEN    object            8367          2            0.02
3      NUMBER_OF_OWNERS    object            8369          0            0.00
4   YEAR_OF_MANUFACTURE   float64            8349         20            0.24
5               MILEAGE    object            8082        287            3.43
6             FUEL_TYPE    object            8369          0            0.00
7            MODEL_YEAR     int64            8369          0            0.00
8    INSURANCE_VALIDITY    object            8365          4            0.05
9       NUMBER_OF_GEARS    object            8263        106            1.27
10         TRANSMISSION    object            8369          0            0.00
11                SEATS    object            8363          6            0.07

***b)Handling Missing Values:***

 Identify and fill or remove missing values in the dataset. 

i)	For numerical columns, use techniques like mean, median, or mode imputation.

ii)	For categorical columns, use mode imputation or create a new category for missing values.


In [32]:
import pandas as pd
import numpy as np
import os

# Load the dataset
df = pd.read_csv("csv_files/necessary_wanted_cols.csv")

# Function to print missing value statistics
def print_missing_stats(df, title="Before Imputation"):
    print(f"\n🔍 {title} Missing Values Summary:")
    print("="*60)
    missing_data = df.isnull().sum()
    total_rows = len(df)
    missing_percent = (missing_data / total_rows) * 100

    stats_df = pd.DataFrame({
        'Missing Values': missing_data,
        '% Missing': missing_percent.round(2)
    })

    print(stats_df[stats_df['Missing Values'] > 0].sort_values('% Missing', ascending=False))
    print("="*60)
    print(f"Total rows in dataset: {total_rows}\n")

# Initial missing value analysis
print_missing_stats(df)

# Create a copy for tracking changes
df_cleaned_filled = df.copy()

# Dictionary to store imputation details
imputation_report = {}

# Handle numerical columns
numerical_cols = ['YEAR_OF_MANUFACTURE']
for col in numerical_cols:
    if col in df_cleaned_filled.columns:
        before = df_cleaned_filled[col].isnull().sum()
        median_val = df_cleaned_filled[col].median()
        df_cleaned_filled[col].fillna(median_val, inplace=True)
        after = df_cleaned_filled[col].isnull().sum()

        imputation_report[col] = {
            'type': 'numerical',
            'before': before,
            'filled': before - after,
            'method': 'median',
            'value': median_val,
            'justification': 'Median is robust against outliers in manufacturing years'
        }

# Handle categorical columns
categorical_cols = ['COLOR', 'REGISTRATION_YEAR', 'INSURANCE_VALIDITY',
                    'NUMBER_OF_GEARS', 'SEATS', 'ENGINE DISPLACEMENT', 'MILEAGE']

for col in categorical_cols:
    if col in df_cleaned_filled.columns:
        before = df_cleaned_filled[col].isnull().sum()
        mode_val = df_cleaned_filled[col].mode()[0]
        df_cleaned_filled[col].fillna(mode_val, inplace=True)
        after = df_cleaned_filled[col].isnull().sum()

        imputation_report[col] = {
            'type': 'categorical',
            'before': before,
            'filled': before - after,
            'method': 'mode',
            'value': mode_val,
            'justification': 'Most frequent value is appropriate for categorical data'
        }

# Special handling for 'MILEAGE' column
if 'MILEAGE' in df_cleaned_filled.columns:
    # Extract numerical part from Mileage (e.g., '23.1 kmpl' -> 23.1)
    df_cleaned_filled['MILEAGE'] = df_cleaned_filled['MILEAGE'].str.extract('(\d+\.?\d*)').astype(float)

    before = df_cleaned_filled['MILEAGE'].isnull().sum()
    median_mileage = df_cleaned_filled['MILEAGE'].median()
    df_cleaned_filled['MILEAGE'].fillna(median_mileage, inplace=True)
    after = df_cleaned_filled['MILEAGE'].isnull().sum()

    imputation_report['MILEAGE'] = {
        'type': 'converted numerical',
        'before': before,
        'filled': before - after,
        'method': 'median',
        'value': median_mileage,
        'justification': 'After converting string to numerical, median is robust for mileage values'
    }

# Final missing value analysis
print_missing_stats(df_cleaned_filled, "After Imputation")

# Generate imputation report
print("\n✅ Imputation Summary")
print("="*60)
print("Here's a breakdown of how missing values were handled in your dataset:")
print("-"*60)

if imputation_report:  # Check if the dictionary is not empty
    report_df = pd.DataFrame.from_dict(imputation_report, orient='index')
    # Ensure the expected columns exist before selection
    expected_columns = ['type', 'before', 'filled', 'method', 'value', 'justification']
    existing_columns = report_df.columns.tolist()
    columns_to_select = [col for col in expected_columns if col in existing_columns]

    if columns_to_select:
        report_df = report_df[columns_to_select]
        report_df.columns = ['Type', 'Missing Before', 'Filled', 'Strategy', 'Value Used', 'Justification']
        print(report_df.sort_values('Missing Before', ascending=False))
    else:
        print("No missing values were imputed, so no imputation report to display.")
else:
    print("No missing values were found in the dataset.")

print("="*60)

# Detect and print data types
print("\n🔎 Final Data Types:")
print("="*60)
print(df_cleaned_filled.dtypes)
print("="*60)

# Create a copy to detect data types without modifying the original
df_detected_dtypes = df_cleaned_filled.copy().convert_dtypes()

print("Original DataFrame dtypes:")
print(df_cleaned_filled.dtypes)
print("\nDetected dtypes (without saving):")
print(df_detected_dtypes.dtypes)
print("\nAutomatically detected the datatypes (in the copy)")

#  replace it here

# Save the cleaned data with detected data types
os.makedirs("csv_files", exist_ok=True)
output_path = "csv_files/cars_filled_removed.csv"
df_detected_dtypes.to_csv(output_path, index=False)

# print(f"\n✅ Cleaned dataset saved to: {output_path}")


🔍 Before Imputation Missing Values Summary:
                     Missing Values  % Missing
MILEAGE                         287       3.43
NUMBER_OF_GEARS                 106       1.27
YEAR_OF_MANUFACTURE              20       0.24
SEATS                             6       0.07
BODY_TYPE                         4       0.05
INSURANCE_VALIDITY                4       0.05
KILOMETERS_DRIVEN                 2       0.02
Total rows in dataset: 8369


🔍 After Imputation Missing Values Summary:
                   Missing Values  % Missing
BODY_TYPE                       4       0.05
KILOMETERS_DRIVEN               2       0.02
Total rows in dataset: 8369


✅ Imputation Summary
Here's a breakdown of how missing values were handled in your dataset:
------------------------------------------------------------
                                    Type  Missing Before  Filled Strategy  \
NUMBER_OF_GEARS              categorical             106     106     mode   
YEAR_OF_MANUFACTURE            num

C:\Users\USER\AppData\Local\Temp\ipykernel_25204\3120422852.py:40: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned_filled[col].fillna(median_val, inplace=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_25204\3120422852.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.



In [33]:
nan_summary = pd.DataFrame({
    'Column': df_detected_dtypes .columns,
    'Data Type': df_detected_dtypes .dtypes,
    'Non-Null Count': df_detected_dtypes.notna().sum(),
    'NaN Count': df_detected_dtypes .isna().sum(),
    'NaN Percentage': (df_detected_dtypes.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column       Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 PRICE  string[python]            8369          0            0.00
1             BODY_TYPE  string[python]            8365          4            0.05
2     KILOMETERS_DRIVEN  string[python]            8367          2            0.02
3      NUMBER_OF_OWNERS  string[python]            8369          0            0.00
4   YEAR_OF_MANUFACTURE           Int64            8369          0            0.00
5               MILEAGE         Float64            8369          0            0.00
6             FUEL_TYPE  string[python]            8369          0            0.00
7            MODEL_YEAR           Int64            8369          0            0.00
8    INSURANCE_VALIDITY  string[python]            8369          0            0.00
9       NUMBER_OF_GEARS  string[python]            8369          0            0.00
10         TRANSMISSION  string[python]            8369          0            0.00
11  

In [ ]:
# Define the columns to check for NaNs or blanks
columns_to_check = [
    'PRICE',
    'BODY_TYPE',
    'KILOMETERS_DRIVEN',
    'NUMBER_OF_OWNERS',
    'YEAR_OF_MANUFACTURE',
    'MILEAGE',
    'FUEL_TYPE',
    'REGISTRATION_YEAR',
    'MODEL_YEAR',
    'INSURANCE_VALIDITY',
    'NUMBER_OF_GEARS',
    'TRANSMISSION',
    'SEATS',
    'CITY_NAME'
]

# Identify rows with NaN or blank in the specified columns
rows_to_drop = set()
for col in columns_to_check:
    if col in df.columns:  # Assuming your DataFrame is named 'df'
        nan_rows = df[df[col].isna()].index
        blank_rows = df[df[col].astype(str).str.strip() == ''].index
        rows_to_drop.update(nan_rows)
        rows_to_drop.update(blank_rows)

rows_to_drop = sorted(list(rows_to_drop))

# Drop the identified rows
if rows_to_drop:
    print(f"\n⚠️ Dropping {len(rows_to_drop)} rows with NaN or blank values in the specified columns.")
    df.drop(index=rows_to_drop, inplace=True)  # Assuming your DataFrame is named 'df'
else:
    print("\n✅ No rows with NaN or blank values found in the specified columns.")

check_please

In [8]:
import pandas as pd
import numpy as np
import os

# Load the dataset
df = pd.read_csv("csv_files/necessary_wanted_cols.csv")

# Function to print missing value statistics
def print_missing_stats(df, title="Before Imputation"):
    print(f"\n🔍 {title} Missing Values Summary:")
    print("="*60)
    missing_data = df.isnull().sum()
    total_rows = len(df)
    missing_percent = (missing_data / total_rows) * 100

    stats_df = pd.DataFrame({
        'Missing Values': missing_data,
        '% Missing': missing_percent.round(2)
    })

    print(stats_df[stats_df['Missing Values'] > 0].sort_values('% Missing', ascending=False))
    print("="*60)
    print(f"Total rows in dataset: {total_rows}\n")

# Initial missing value analysis
print_missing_stats(df)

# Create a copy for tracking changes
df_cleaned_filled = df.copy()

# Dictionary to store imputation details
imputation_report = {}

# Handle numerical columns
numerical_cols_to_fill = ['YEAR_OF_MANUFACTURE', 'KILOMETERS_DRIVEN']
for col in numerical_cols_to_fill:
    if col in df_cleaned_filled.columns:
        # Attempt to convert to numeric, errors='coerce' will turn non-numeric to NaN
        df_cleaned_filled[col] = pd.to_numeric(df_cleaned_filled[col], errors='coerce')
        before = df_cleaned_filled[col].isnull().sum()
        median_val = df_cleaned_filled[col].median()
        df_cleaned_filled[col].fillna(median_val, inplace=True)
        after = df_cleaned_filled[col].isnull().sum()

        imputation_report[col] = {
            'type': 'numerical',
            'before': before,
            'filled': before - after,
            'method': 'median',
            'value': median_val,
            'justification': f'Converted to numeric and filled with median due to potential non-numeric entries.'
        }

# Handle categorical columns
categorical_cols_to_fill = ['COLOR', 'REGISTRATION_YEAR', 'INSURANCE_VALIDITY',
                            'NUMBER_OF_GEARS', 'SEATS', 'ENGINE DISPLACEMENT', 'MILEAGE', 'BODY_TYPE']

for col in categorical_cols_to_fill:
    if col in df_cleaned_filled.columns:
        before = df_cleaned_filled[col].isnull().sum()
        # Handle potential errors if mode is empty
        try:
            mode_val = df_cleaned_filled[col].mode()[0]
            df_cleaned_filled[col].fillna(mode_val, inplace=True)
            after = df_cleaned_filled[col].isnull().sum()
            imputation_report[col] = {
                'type': 'categorical',
                'before': before,
                'filled': before - after,
                'method': 'mode',
                'value': mode_val,
                'justification': 'Filled with the most frequent value.'
            }
        except IndexError:
            print(f"Warning: Cannot fill NaN in '{col}' as the mode is not available (all values might be NaN).")
            imputation_report[col] = {
                'type': 'categorical',
                'before': before,
                'filled': 0,
                'method': 'none',
                'value': None,
                'justification': 'Mode not available.'
            }

# Special handling for 'MILEAGE' column
if 'MILEAGE' in df_cleaned_filled.columns:
    # Extract numerical part from Mileage (e.g., '23.1 kmpl' -> 23.1)
    df_cleaned_filled['MILEAGE'] = df_cleaned_filled['MILEAGE'].str.extract('(\d+\.?\d*)').astype(float)

    before = df_cleaned_filled['MILEAGE'].isnull().sum()
    median_mileage = df_cleaned_filled['MILEAGE'].median()
    df_cleaned_filled['MILEAGE'].fillna(median_mileage, inplace=True)
    after = df_cleaned_filled['MILEAGE'].isnull().sum()

    imputation_report['MILEAGE'] = {
        'type': 'converted numerical',
        'before': before,
        'filled': before - after,
        'method': 'median',
        'value': median_mileage,
        'justification': 'After converting string to numerical, median is robust for mileage values'
    }

# Final missing value analysis
print_missing_stats(df_cleaned_filled, "After Imputation")

# Generate imputation report
print("\n✅ Imputation Summary")
print("="*60)
print("Here's a breakdown of how missing values were handled in your dataset:")
print("-"*60)

if imputation_report:
    report_df = pd.DataFrame.from_dict(imputation_report, orient='index')
    expected_columns = ['type', 'before', 'filled', 'method', 'value', 'justification']
    existing_columns = report_df.columns.tolist()
    columns_to_select = [col for col in expected_columns if col in existing_columns]

    if columns_to_select:
        report_df = report_df[columns_to_select]
        report_df.columns = ['Type', 'Missing Before', 'Filled', 'Strategy', 'Value Used', 'Justification']
        print(report_df.sort_values('Missing Before', ascending=False))
    else:
        print("No missing values were imputed, so no imputation report to display.")
else:
    print("No missing values were found in the dataset.")

print("="*60)

# Detect and print data types
print("\n🔎 Final Data Types:")
print("="*60)
print(df_cleaned_filled.dtypes)
print("="*60)

# Create a copy to detect data types without modifying the original
df_detected_dtypes = df_cleaned_filled.copy().convert_dtypes()

print("Original DataFrame dtypes:")
print(df_cleaned_filled.dtypes)
print("\nDetected dtypes (without saving):")
print(df_detected_dtypes.dtypes)
print("\nAutomatically detected the datatypes (in the copy)")

# Save the cleaned data with detected data types
os.makedirs("csv_files", exist_ok=True)
output_path = "csv_files/cars_filled_removed.csv"
df_detected_dtypes.to_csv(output_path, index=False)

print(f"\n✅ Cleaned dataset saved to: {output_path}")


🔍 Before Imputation Missing Values Summary:
                     Missing Values  % Missing
MILEAGE                         287       3.43
NUMBER_OF_GEARS                 106       1.27
YEAR_OF_MANUFACTURE              20       0.24
SEATS                             6       0.07
BODY_TYPE                         4       0.05
INSURANCE_VALIDITY                4       0.05
KILOMETERS_DRIVEN                 2       0.02
Total rows in dataset: 8369


🔍 After Imputation Missing Values Summary:
                   Missing Values  % Missing
KILOMETERS_DRIVEN            8369      100.0
Total rows in dataset: 8369


✅ Imputation Summary
Here's a breakdown of how missing values were handled in your dataset:
------------------------------------------------------------
                                    Type  Missing Before  Filled Strategy  \
KILOMETERS_DRIVEN              numerical            8369       0   median   
NUMBER_OF_GEARS              categorical             106     106     mode   
YE

C:\Users\USER\AppData\Local\Temp\ipykernel_25204\3501016970.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned_filled[col].fillna(median_val, inplace=True)
c:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\USER\AppData\Local\Temp\ipykernel_25204\3501016970.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series throu

In [10]:
import pandas as pd
import numpy as np
import os

# Load the dataset
df = pd.read_csv("csv_files/necessary_wanted_cols.csv")

# Print info before any processing
print("\n--- Info Before Processing 'KILOMETERS_DRIVEN' ---")
print(df['KILOMETERS_DRIVEN'].info())
print("\n--- Unique Values in 'KILOMETERS_DRIVEN' ---")
print(df['KILOMETERS_DRIVEN'].unique()[:50]) # Display the first 50 unique values
print("\n--- Value Counts of 'KILOMETERS_DRIVEN' ---")
print(df['KILOMETERS_DRIVEN'].value_counts().head(20)) # Display the top 20 most frequent values

# Create a copy for tracking changes
df_cleaned_filled = df.copy()

# Dictionary to store imputation details
imputation_report = {}

# Handle numerical columns
numerical_cols_to_fill = ['YEAR_OF_MANUFACTURE', 'KILOMETERS_DRIVEN']
for col in numerical_cols_to_fill:
    if col in df_cleaned_filled.columns:
        print(f"\n--- Processing column: {col} ---")
        # Attempt to convert to numeric, errors='coerce' will turn non-numeric to NaN
        df_cleaned_filled[col] = pd.to_numeric(df_cleaned_filled[col], errors='coerce')
        before = df_cleaned_filled[col].isnull().sum()
        median_val = df_cleaned_filled[col].median()
        df_cleaned_filled[col].fillna(median_val, inplace=True)
        after = df_cleaned_filled[col].isnull().sum()

        imputation_report[col] = {
            'type': 'numerical',
            'before': before,
            'filled': before - after,
            'method': 'median',
            'value': median_val,
            'justification': f'Converted to numeric and filled with median due to potential non-numeric entries.'
        }

# Handle categorical columns
categorical_cols_to_fill = ['COLOR', 'REGISTRATION_YEAR', 'INSURANCE_VALIDITY',
                            'NUMBER_OF_GEARS', 'SEATS', 'ENGINE DISPLACEMENT', 'MILEAGE', 'BODY_TYPE']

for col in categorical_cols_to_fill:
    if col in df_cleaned_filled.columns:
        before = df_cleaned_filled[col].isnull().sum()
        try:
            mode_val = df_cleaned_filled[col].mode()[0]
            df_cleaned_filled[col].fillna(mode_val, inplace=True)
            after = df_cleaned_filled[col].isnull().sum()
            imputation_report[col] = {
                'type': 'categorical',
                'before': before,
                'filled': before - after,
                'method': 'mode',
                'value': mode_val,
                'justification': 'Filled with the most frequent value.'
            }
        except IndexError:
            print(f"Warning: Cannot fill NaN in '{col}' as the mode is not available (all values might be NaN).")
            imputation_report[col] = {
                'type': 'categorical',
                'before': before,
                'filled': 0,
                'method': 'none',
                'value': None,
                'justification': 'Mode not available.'
            }

# Special handling for 'MILEAGE' column
if 'MILEAGE' in df_cleaned_filled.columns:
    # Extract numerical part from Mileage (e.g., '23.1 kmpl' -> 23.1)
    df_cleaned_filled['MILEAGE'] = df_cleaned_filled['MILEAGE'].str.extract('(\d+\.?\d*)').astype(float)

    before = df_cleaned_filled['MILEAGE'].isnull().sum()
    median_mileage = df_cleaned_filled['MILEAGE'].median()
    df_cleaned_filled['MILEAGE'].fillna(median_mileage, inplace=True)
    after = df_cleaned_filled['MILEAGE'].isnull().sum()

    imputation_report['MILEAGE'] = {
        'type': 'converted numerical',
        'before': before,
        'filled': before - after,
        'method': 'median',
        'value': median_mileage,
        'justification': 'After converting string to numerical, median is robust for mileage values'
    }

# Final missing value analysis
print_missing_stats(df_cleaned_filled, "After Imputation")

# Generate imputation report
print("\n✅ Imputation Summary")
print("="*60)
print("Here's a breakdown of how missing values were handled in your dataset:")
print("-"*60)

if imputation_report:
    report_df = pd.DataFrame.from_dict(imputation_report, orient='index')
    expected_columns = ['type', 'before', 'filled', 'method', 'value', 'justification']
    existing_columns = report_df.columns.tolist()
    columns_to_select = [col for col in expected_columns if col in existing_columns]

    if columns_to_select:
        report_df = report_df[columns_to_select]
        report_df.columns = ['Type', 'Missing Before', 'Filled', 'Strategy', 'Value Used', 'Justification']
        print(report_df.sort_values('Missing Before', ascending=False))
    else:
        print("No missing values were imputed, so no imputation report to display.")
else:
    print("No missing values were found in the dataset.")

print("="*60)

# Detect and print data types
print("\n🔎 Final Data Types:")
print("="*60)
print(df_cleaned_filled.dtypes)
print("="*60)

# Create a copy to detect data types without modifying the original
df_detected_dtypes = df_cleaned_filled.copy().convert_dtypes()

print("Original DataFrame dtypes:")
print(df_cleaned_filled.dtypes)
print("\nDetected dtypes (without saving):")
print(df_detected_dtypes.dtypes)
print("\nAutomatically detected the datatypes (in the copy)")

# Save the cleaned data with detected data types
os.makedirs("csv_files", exist_ok=True)
output_path = "csv_files/cars_filled_removed.csv"
df_detected_dtypes.to_csv(output_path, index=False)

print(f"\n✅ Cleaned dataset saved to: {output_path}")


--- Info Before Processing 'KILOMETERS_DRIVEN' ---
<class 'pandas.core.series.Series'>
RangeIndex: 8369 entries, 0 to 8368
Series name: KILOMETERS_DRIVEN
Non-Null Count  Dtype 
--------------  ----- 
8367 non-null   object
dtypes: object(1)
memory usage: 65.5+ KB
None

--- Unique Values in 'KILOMETERS_DRIVEN' ---
['1,20,000 Kms' '32,706 Kms' '11,949 Kms' '17,794 Kms' '60,000 Kms'
 '20,000 Kms' '37,772 Kms' '30,000 Kms' '37,000 Kms' '50,000 Kms'
 '24,887 Kms' '23,000 Kms' '48,928 Kms' '1,08,862 Kms' '80,000 Kms'
 '16,000 Kms' '65,376 Kms' '10,000 Kms' '67,000 Kms' '1,00,000 Kms'
 '18,083 Kms' '57,634 Kms' '1,10,000 Kms' '23,447 Kms' '33,000 Kms'
 '87,667 Kms' '40,000 Kms' '79,012 Kms' '31,344 Kms' '12,434 Kms'
 '12,000 Kms' '40,987 Kms' '18,157 Kms' '89,667 Kms' '56,997 Kms'
 '59,000 Kms' '18,835 Kms' '60,932 Kms' '33,539 Kms' '66,837 Kms'
 '400 Kms' '44,285 Kms' '25,000 Kms' '36,735 Kms' '79,000 Kms'
 '53,645 Kms' '53,982 Kms' '37,741 Kms' '17,367 Kms' '94,000 Kms']

--- Value Counts 

C:\Users\USER\AppData\Local\Temp\ipykernel_25204\3937719871.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned_filled[col].fillna(median_val, inplace=True)
c:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\USER\AppData\Local\Temp\ipykernel_25204\3937719871.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series throu

In [12]:
nan_summary = pd.DataFrame({
    'Column': df_detected_dtypes .columns,
    'Data Type': df_detected_dtypes .dtypes,
    'Non-Null Count': df_detected_dtypes.notna().sum(),
    'NaN Count': df_detected_dtypes .isna().sum(),
    'NaN Percentage': (df_detected_dtypes.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column       Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 PRICE  string[python]            8369          0             0.0
1             BODY_TYPE  string[python]            8369          0             0.0
2     KILOMETERS_DRIVEN           Int64               0       8369           100.0
3      NUMBER_OF_OWNERS  string[python]            8369          0             0.0
4   YEAR_OF_MANUFACTURE           Int64            8369          0             0.0
5               MILEAGE         Float64            8369          0             0.0
6             FUEL_TYPE  string[python]            8369          0             0.0
7            MODEL_YEAR           Int64            8369          0             0.0
8    INSURANCE_VALIDITY  string[python]            8369          0             0.0
9       NUMBER_OF_GEARS  string[python]            8369          0             0.0
10         TRANSMISSION  string[python]            8369          0             0.0
11  

***c)	Standardising Data Formats:***

i)	Check for all data types and do the necessary steps to keep the data in the correct format.

(1)	Eg. If a data point has string formats like 70 kms, then remove the unit ‘kms’ and change the data type from string to integers.


In [6]:
df_detected_dtypes.head()

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,₹ 4 Lakh,Hatchback,"1,20,000 Kms",3rd Owner,2015,23.1,KA51,Petrol,2015,2015,Third Party insurance,5 Speed,Manual,5 Seats,bangalore,998 cc
1,₹ 8.11 Lakh,SUV,"32,706 Kms",2nd Owner,2018,17.0,KA05,Petrol,Feb 2018,2018,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1497 cc
2,₹ 5.85 Lakh,Hatchback,"11,949 Kms",1st Owner,2018,23.84,KA03,Petrol,Sept 2018,2018,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1199 cc
3,₹ 4.62 Lakh,Sedan,"17,794 Kms",1st Owner,2014,19.1,KA53,Petrol,Dec 2014,2014,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1197 cc
4,₹ 7.90 Lakh,SUV,"60,000 Kms",1st Owner,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5 Speed,Manual,5 Seats,bangalore,1248 cc


| Transmission Type | Gear Number        | Justification                                                                                                                                                              |
|-------------------|--------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| CVT               | Theoretically Infinite | Uses a belt and pulley (or cone) system that continuously adjusts the ratio, offering a seamless and theoretically infinite number of gear ratios within its operating range. |
| Direct Drive      | Typically One (1:1) | Represents a single, direct connection where the input and output shafts rotate at the same speed, with no gear reduction or multiplication.                               |
| IVT               | Theoretically Infinite | Hyundai/Kia's marketing term for their CVT technology, functioning on the same principle of continuous ratio adjustment via a belt and pulley system.                   |
| iMT               | Fixed (e.g., 5 or 6) | A manual transmission with a traditional set of physical gears. The electronic system automates the clutch, but the number of selectable gears remains fixed.             |
| AGS               | Fixed (e.g., 5)    | An automated manual transmission that uses a traditional set of physical gears. The system automates both clutch operation and gear selection, but the number of gears is fixed. |

In [10]:
import pandas as pd
from decimal import Decimal
import os
import re


try:
    df_standardised = pd.read_csv("output/banglore_filled_removed.csv")  # Ensure correct file name
except FileNotFoundError:
    print("Error: The file 'output/banglore_filled_removed.csv' was not found.")
    exit()


def clean_numeric_column(series, pattern=None, dtype=float):
    """Helper function to clean numeric columns"""
    try:
        if series.dtype == object:
            if pattern:
                extracted = series.str.extract(pattern)[0]
            else:
                extracted = series.astype(str).str.replace(r'[^\d\.]', '', regex=True) # Keep decimals
            if dtype == 'Int64':
                return pd.to_numeric(extracted, errors='coerce').astype('Int64')
            return pd.to_numeric(extracted, errors='coerce')
        return series
    except Exception as e:
        print(f"⚠️ Warning cleaning column {series.name}: {e}")
        return series
    
def clean_numeric_series(series, dtype=float, pattern=None):
    """Clean a pandas Series containing numeric values"""
    try:
        if not pd.api.types.is_string_dtype(series):
            series = series.astype(str)
            
        # Remove commas and extract numeric values
        cleaned = series.str.replace(',', '')
        
        if pattern:
            extracted = cleaned.str.extract(pattern)[0]
        else:
            if dtype == float:
                extracted = cleaned.str.extract(r'([\d\.]+)')[0]
            else:
                extracted = cleaned.str.extract(r'(\d+)')[0]
        
        # Convert to appropriate type
        if dtype == 'Int64':
            return pd.to_numeric(extracted, errors='coerce').astype('Int64')
        return pd.to_numeric(extracted, errors='coerce').astype(dtype)
    except Exception as e:
        print(f"⚠️ Warning cleaning numeric series: {e}")
        return series
    
def clean_kms_driven(kms_str):
    """
    Cleans the 'Kms Driven' string by removing non-numeric characters
    and converting it to an integer.
    Handles cases with commas and the "Kms" suffix.
    Returns the cleaned number as an integer or None if cleaning fails.
    """
    if isinstance(kms_str, (int, float)):
        return int(kms_str)  # Already numeric

    cleaned_str = re.sub(r'[^\d]', '', str(kms_str))  # Remove non-digits
    if cleaned_str:
        return int(cleaned_str)
    return None

if 'Kms Driven' in df_standardised.columns:
    df_standardised['Kms Driven'] = df_standardised['Kms Driven'].apply(clean_kms_driven)
    print("✅ Processed 'Kms Driven' column - removed non-numeric characters and converted to integer")

def convert_price_to_numeric(price_str):
    """
    Converts price strings in '₹ X.XX Lakh', '₹ X Lakh', or '₹ X.XX Crore' format to numeric INR.
    Handles potential non-numeric values by returning NaN.
    """
    if isinstance(price_str, (int, float, Decimal)):
        return float(price_str)  # Already numeric

    price_str = str(price_str).strip().replace(',', '')  # Remove leading/trailing whitespace and commas

    lakh_match = re.search(r'₹\s*(\d+(\.\d+)?)\s*Lakh', price_str, re.IGNORECASE)
    if lakh_match:
        return float(Decimal(lakh_match.group(1)) * Decimal('100000'))

    crore_match = re.search(r'₹\s*(\d+(\.\d+)?)\s*Crore', price_str, re.IGNORECASE)
    if crore_match:
        return float(Decimal(crore_match.group(1)) * Decimal('10000000'))

    # Handle cases with just numeric values (after removing '₹' and spaces)
    numeric_only = price_str.replace('₹', '').strip()
    if numeric_only.isdigit() or ('.' in numeric_only and all(c.isdigit() or c == '.' for c in numeric_only)):
        try:
            return float(numeric_only)
        except ValueError:
            return pd.NA  # Return NaN for invalid numeric strings

    return pd.NA  # Return NaN for non-convertible strings

if 'price' in df_standardised.columns:
    original_prices = df_standardised['price'].copy()
    df_standardised['price_numeric'] = df_standardised['price'].apply(convert_price_to_numeric)

    # Count NaN values after conversion
    nan_count = df_standardised['price_numeric'].isna().sum()

    # Identify non-convertible original values
    non_convertible_original = original_prices[df_standardised['price_numeric'].isna()].unique().tolist()

    # Update the original 'price' column with the numeric conversions
    df_standardised['price'] = df_standardised['price_numeric']

    print("✅ Processed 'price' column - attempted to convert 'Lakh' and 'Crore' values to numeric INR")
    print(f"Number of NaN values in 'price' column after conversion: {nan_count}")
    if non_convertible_original:
        print(f"\n⚠️ The following original values in the 'price' column could not be converted to numeric: {non_convertible_original}")
    else:
        print("\n🎉 All identified price values were successfully converted to numeric.")

    # Optionally, you can drop the temporary 'price_numeric' column
    if 'price_numeric' in df_standardised.columns:
        df_standardised.drop(columns=['price_numeric'], inplace=True)



def extract_gear_speeds_v4_modified(gear_box_str):
    """
    Extracts the number of speeds from a Gear Box string.
    Handles various formats including explicit speed numbers and common transmission types,
    assigning specific numerical values to CVT, Direct Drive, IVT, iMT, and AGS.
    Returns an integer representing the number of speeds or a specific code for other types.
    """
    if isinstance(gear_box_str, (int, float)):
        return int(gear_box_str)

    gear_box_str = str(gear_box_str).strip().lower()

    # Explicit number of speeds
    speed_match = re.search(r'(\d+)\s*speed', gear_box_str)
    if speed_match:
        return int(speed_match.group(1))

    speed_match_hyphen = re.search(r'(\d+)-speed', gear_box_str)
    if speed_match_hyphen:
        return int(speed_match_hyphen.group(1))

    # Leading number (e.g., 8G-DCT)
    leading_number_match = re.search(r'^(\d+)', gear_box_str)
    if leading_number_match:
        return int(leading_number_match.group(1))

    # Handle specific transmission types with assigned numerical values
    if 'cvt' in gear_box_str or 'ivt' in gear_box_str:
        return 0  # Or -1 to represent continuous
    elif 'direct drive' in gear_box_str:
        return 1
    elif 'imt' in gear_box_str:
        return 5  # Or 6, depending on the common range
    elif 'ags' in gear_box_str:
        return 5  # Or 6, depending on the common range

    # Handle manual transmissions (explicitly mention number of speeds)
    manual_match = re.search(r'(\d+)\s*speed\s*manual(?: transmission)?', gear_box_str)
    if manual_match:
        return int(manual_match.group(1))
    manual_match_short = re.search(r'five speed manual', gear_box_str) # Specific case
    if manual_match_short:
        return 5

    return pd.NA  # Return NaN for truly unidentifiable cases

if 'Gear Box' in df_standardised.columns:
    original_gear_box = df_standardised['Gear Box'].copy()
    df_standardised['Gear Box_numeric'] = df_standardised['Gear Box'].apply(extract_gear_speeds_v4_modified)

    # Update the original 'Gear Box' column with the numeric conversions
    df_standardised['Gear Box'] = df_standardised['Gear Box_numeric']

    print("✅ Processed 'Gear Box' column - attempted to extract the number of speeds (version 4 - modified for ML)")
    print(f"Number of NaN values in 'Gear Box' column after extraction: {df_standardised['Gear Box'].isnull().sum()}")
    if df_standardised['Gear Box'].isnull().sum() > 0:
        print(f"\n⚠️ There are still NaN values in the 'Gear Box' column for truly unidentifiable strings.")
    else:
        print("\n🎉 All 'Gear Box' values were successfully converted to a numeric representation.")

    #Optionally, you might not want to drop the temporary column for inspection
    if 'Gear Box_numeric' in df_standardised.columns:
        df_standardised.drop(columns=['Gear Box_numeric'], inplace=True)


def clean_and_transform(df_standardised):
    """Perform all cleaning and transformation operations"""
    try:
        df_transformed = df_standardised.copy()

        # ===== PROCESS KM COLUMN =====
        if 'Kms Driven' in df_standardised.columns:
            df_standardised['Kms Driven'] = clean_numeric_column(df_standardised['Kms Driven'], 'Int64')
            print("✅ Processed 'Kms Driven' column - removed commas and converted to integer")

        # ===== STANDARD TRANSFORMATIONS =====
        transformations = [
            ('owner', r'(\d+)', 'Int64'),
            ('Registration Year', r'(\d+)', 'Int64'),
            ('Seats', r'(\d+)', 'Int64'),
            ('Engine Displacement', r'(\d+)', 'Int64'),
            ('Mileage', r'(\d+\.?\d*)', float) # Extract numerical mileage
        ]

        for col, pattern, action in transformations:
            if col in df_transformed.columns:
                try:
                    if callable(action):
                        if pattern:
                            match = df_transformed[col].str.extract(pattern)
                            df_transformed[col] = match[0].apply(action) if not match.empty else None
                        else:
                            df_transformed[col] = df_transformed[col].apply(action)
                    else:
                        df_transformed[col] = clean_numeric_column(df_transformed[col], pattern, action)
                except Exception as e:
                    print(f"⚠️ Warning processing column {col}: {e}")

        return df_transformed
    except Exception as e:
        print(f"❌ Error in transformation: {e}")
        return None




df_standardised = clean_and_transform(df_standardised)

# Assuming df_standardised is your DataFrame

year_columns_to_convert = ['Year of Manufacture', 'Registration Year', 'modelYear']

for col in year_columns_to_convert:
    if col in df_standardised.columns:
        try:
            df_standardised[col] = pd.to_datetime(df_standardised[col], format='%Y').dt.to_period('Y')
            print(f"Converted '{col}' to Period[Y]")
        except ValueError:
            print(f"Could not convert '{col}' to Period[Y], keeping original type.")

print("\nDataFrame dtypes after converting year columns:")
print(df_standardised.dtypes)

print("\nDataFrame with Period[Y] dtype:")
print(df_standardised[year_columns_to_convert].head())


# Create a copy to detect data types without modifying the original
df_standardised_dtypes = df_standardised.copy().convert_dtypes()

print("Original DataFrame dtypes:")
print(df_standardised.dtypes)
print("\nDetected dtypes (without saving):")
print(df_standardised_dtypes.dtypes)
print("\nAutomatically detected the datatypes (in the copy)")


if df_standardised_dtypes is not None:
    print("\n the datatypes after transformation")
    print(df_standardised_dtypes.dtypes)
    df_standardised_dtypes.head()

    # Save the df_Standardising data
    os.makedirs("output", exist_ok=True)
    output_path = "output/banglore_standardised.csv"
    df_standardised_dtypes.to_csv(output_path, index=False)
    print(f"\n✅ Cleaned dataset saved to: {output_path}")


✅ Processed 'Kms Driven' column - removed non-numeric characters and converted to integer
✅ Processed 'price' column - attempted to convert 'Lakh' and 'Crore' values to numeric INR
Number of NaN values in 'price' column after conversion: 0

🎉 All identified price values were successfully converted to numeric.
✅ Processed 'Gear Box' column - attempted to extract the number of speeds (version 4 - modified for ML)
Number of NaN values in 'Gear Box' column after extraction: 0

🎉 All 'Gear Box' values were successfully converted to a numeric representation.
✅ Processed 'Kms Driven' column - removed commas and converted to integer
⚠️ Warning processing column Mileage: Can only use .str accessor with string values!
Converted 'Year of Manufacture' to Period[Y]
Converted 'Registration Year' to Period[Y]
Converted 'modelYear' to Period[Y]

DataFrame dtypes after converting year columns:
price                        float64
bt                            object
Kms Driven                     int64

In [12]:
df_standardised_dtypes.head(10)

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,400000,Hatchback,120000,3,2015,23.1,KA51,Petrol,2015,2015,Third Party insurance,5,Manual,5,bangalore,998
1,811000,SUV,32706,2,2018,17.0,KA05,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1497
2,585000,Hatchback,11949,1,2018,23.84,KA03,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1199
3,462000,Sedan,17794,1,2014,19.1,KA53,Petrol,2014,2014,Comprehensive,5,Manual,5,bangalore,1197
4,790000,SUV,60000,1,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5,Manual,5,bangalore,1248
5,1900000,SUV,20000,1,2020,17.1,KA04,Diesel,2020,2020,Third Party insurance,6,Manual,5,bangalore,1956
6,345000,Hatchback,37772,1,2017,20.63,KA05,Petrol,2017,2017,Comprehensive,5,Manual,5,bangalore,1198
7,1200000,SUV,30000,1,2021,18.15,KA51,Petrol,2021,2021,Third Party insurance,7,Automatic,5,bangalore,998
8,960000,Sedan,37000,1,2018,20.28,KA03,Petrol,2018,2018,Comprehensive,4,Automatic,5,bangalore,1462
9,585000,Hatchback,11949,1,2017,23.84,KA03,Petrol,2018,2017,Comprehensive,5,Manual,5,bangalore,1199


In [13]:
nan_summary = pd.DataFrame({
    'Column': df_standardised_dtypes .columns,
    'Data Type': df_standardised_dtypes .dtypes,
    'Non-Null Count': df_standardised_dtypes.notna().sum(),
    'NaN Count': df_standardised_dtypes .isna().sum(),
    'NaN Percentage': (df_standardised_dtypes.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column       Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 price           Int64            1481          0             0.0
1                    bt  string[python]            1481          0             0.0
2            Kms Driven           Int64            1481          0             0.0
3                 owner           Int64            1481          0             0.0
4   Year of Manufacture   period[Y-DEC]            1481          0             0.0
5               Mileage         Float64            1481          0             0.0
6                   RTO  string[python]            1481          0             0.0
7             Fuel Type  string[python]            1481          0             0.0
8     Registration Year   period[Y-DEC]            1481          0             0.0
9             modelYear   period[Y-DEC]            1481          0             0.0
10   Insurance Validity  string[python]            1481          0             0.0
11  

In [ ]:
import pandas as pd
import os

# File path of the input CSV file
input_file_path = r'C:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\output\banglore_standardised.csv'

# Output directory
output_dir = r'C:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\output'
output_file_name = 'Banglore_data_format.csv'
output_file_path = os.path.join(output_dir, output_file_name)

try:
    # Read the CSV file into a Pandas DataFrame
    df = pd.read_csv(input_file_path)

    # Create a mapping dictionary for column name changes
    column_name_mapping = {
        'BODY_TYPE': 'BODY_TYPE',  # Keep as is but ensure consistent capitalization
        'Kms Driven': 'Kilometers_Driven',
        'Gear Box': 'NUMBER_OF_GEARS',
        'City': 'CITY_NAME',
        'modelYear': 'MODEL_YEAR'
    }

    # Function to format column names
    def format_column_name(col_name):
        if col_name in column_name_mapping:
            return column_name_mapping[col_name].upper()
        else:
            return col_name.upper().replace(' ', '_')

    # Apply the formatting function to all column names
    df.columns = [format_column_name(col) for col in df.columns]

    # Save the modified DataFrame to a new CSV file
    df.to_csv(output_file_path, index=False, encoding='utf-8')

    print(f"✅ Successfully formatted column names and saved to: {output_file_path}")

except FileNotFoundError:
    print(f"❌ Error: Input file not found at: {input_file_path}")
except Exception as e:
    print(f"❌ An error occurred: {e}")

***d)	Encoding Categorical Variables: Convert categorical features into numerical values using encoding techniques.***

i)	Use one-hot encoding for nominal categorical variables.

ii)	Use label encoding or ordinal encoding for ordinal categorical variables.


In [3]:
import pandas as pd


file_path=r"C:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\output\banglore_standardised.csv"
df_Banglore_1=pd.read_csv(file_path)

df_Banglore_1.head(10)

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,400000,Hatchback,120000,3,2015,23.10,KA51,Petrol,2015,2015,Third Party insurance,5,Manual,5,bangalore,998
1,811000,SUV,32706,2,2018,17.00,KA05,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1497
2,585000,Hatchback,11949,1,2018,23.84,KA03,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1199
3,462000,Sedan,17794,1,2014,19.10,KA53,Petrol,2014,2014,Comprehensive,5,Manual,5,bangalore,1197
4,790000,SUV,60000,1,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5,Manual,5,bangalore,1248
5,1900000,SUV,20000,1,2020,17.10,KA04,Diesel,2020,2020,Third Party insurance,6,Manual,5,bangalore,1956
6,345000,Hatchback,37772,1,2017,20.63,KA05,Petrol,2017,2017,Comprehensive,5,Manual,5,bangalore,1198
7,1200000,SUV,30000,1,2021,18.15,KA51,Petrol,2021,2021,Third Party insurance,7,Automatic,5,bangalore,998
8,960000,Sedan,37000,1,2018,20.28,KA03,Petrol,2018,2018,Comprehensive,4,Automatic,5,bangalore,1462
9,585000,Hatchback,11949,1,2017,23.84,KA03,Petrol,2018,2017,Comprehensive,5,Manual,5,bangalore,1199


Column Name | Why It's Suitable for Label Encoding
RTO | Regional codes, no meaningful order, many unique values
Insurance Validity | Can be ordinal or label encoded depending on use-case
City | Repetitive categorical field with few values
bt (Body Type) | Few categories, okay for label encoding in some models
Fuel Type | Few values, often used for ordinal, but label works too
Transmission | Manual/Automatic — 2 values, label or ordinal both work

In [20]:
#label Encoding

from sklearn.preprocessing import LabelEncoder

# Create a copy to avoid modifying the original
df_encoded_lablel_encoding = df_Banglore_1.copy()

label_cols = ['RTO', 'Insurance Validity', 'City', 'bt', 'Fuel Type', 'Transmission']

# Apply LabelEncoder to each column
le = LabelEncoder()
for col in label_cols:
    df_encoded_lablel_encoding[col] = le.fit_transform(df_encoded_lablel_encoding[col])

# Preview the result
df_encoded_lablel_encoding.head()


,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,400000,2,120000,3,2015,23.10,45,4,2015,2015,4,5,1,5,0,998
1,811000,6,32706,2,2018,17.00,15,4,2018,2018,5,5,1,5,0,1497
2,585000,2,11949,1,2018,23.84,13,4,2018,2018,5,5,1,5,0,1199
3,462000,7,17794,1,2014,19.10,47,4,2014,2014,5,5,1,5,0,1197
4,790000,6,60000,1,2015,23.65,14,1,2015,2015,4,5,1,5,0,1248


In [26]:
# check for unique values
import pandas as pd

# Assuming df_encoded_lablel_encoding is your DataFrame
if isinstance(df_encoded_lablel_encoding, pd.DataFrame):
    for column in df_encoded_lablel_encoding.columns:
        unique_values = df_encoded_lablel_encoding[column].unique()
        print(f"Unique values in column '{column}':")
        print(unique_values)
        print("-" * 30)
else:
    print("df_encoded_lablel_encoding is not a Pandas DataFrame.")

Unique values in column 'price':
[  400000   811000   585000   462000   790000  1900000   345000  1200000
   960000   690000   682000   825000   595000  1350000  5595000   521000
  1005000   775000  2200000   582000  1090000   457000  4900000   550000
   570000   220000  4145000   861000  1785000   803000  2565000   674000
   349000  1050000  4425000   406000  4965000  1100000   594000   710000
   692000   650000  2090000   411000  3675000   610000  1675000   530000
  7990000   250000   715000  3500000  2250000   625000  2695000   428000
   695000   930000   455000  1195000  2175000   442000   500000  3395000
   537000   750000  1790000   468000   420000   920000   533000   850000
   751000   388000   490000   494000   525000  2890000   365000  1895000
  5990000  2075000   394000   725000   240000   835000  1725000   330000
   990000   755000  1750000  2083000   271000  1280000   436000  1250000
  2225000   475000   600000  2450000  1625000  1650000   249000   630000
  2599000   385000

In [20]:
df_Banglore_1.price.unique()

array([  400000,   811000,   585000,   462000,   790000,  1900000,
         345000,  1200000,   960000,   690000,   682000,   825000,
         595000,  1350000,  5595000,   521000,  1005000,   775000,
        2200000,   582000,  1090000,   457000,  4900000,   550000,
         570000,   220000,  4145000,   861000,  1785000,   803000,
        2565000,   674000,   349000,  1050000,  4425000,   406000,
        4965000,  1100000,   594000,   710000,   692000,   650000,
        2090000,   411000,  3675000,   610000,  1675000,   530000,
        7990000,   250000,   715000,  3500000,  2250000,   625000,
        2695000,   428000,   695000,   930000,   455000,  1195000,
        2175000,   442000,   500000,  3395000,   537000,   750000,
        1790000,   468000,   420000,   920000,   533000,   850000,
         751000,   388000,   490000,   494000,   525000,  2890000,
         365000,  1895000,  5990000,  2075000,   394000,   725000,
         240000,   835000,  1725000,   330000,   990000,   755

In [24]:
df_Banglore_1.owner.unique()

array([3, 2, 1, 4, 5])

Ordinal Encoding

Column | Ordinal Encoding? | Why?
Insurance Validity | ✅ Yes (again) | Has natural order: 'Third Party insurance' < 'Comprehensive'
owner | ✅ Yes | Represents ownership count: 1st owner < 2nd owner < 3rd...
Gear Box | ✅ Yes | Typically higher number = more gears (ordered technically)
Seats | ✅ Yes (Optional) | More seats = larger vehicles — can be ordinal in some contexts

In [21]:

# check for unique values
import pandas as pd

# Assuming df_encoded_lablel_encoding is your DataFrame
if isinstance(df_encoded_lablel_encoding, pd.DataFrame):
    for column in df_encoded_lablel_encoding.columns:
        unique_values = df_encoded_lablel_encoding[column].unique()
        print(f"Unique values in column '{column}':")
        print(unique_values)
        print("-" * 30)
else:
    print("df_encoded_lablel_encoding is not a Pandas DataFrame.")

Unique values in column 'price':
[  400000   811000   585000   462000   790000  1900000   345000  1200000
   960000   690000   682000   825000   595000  1350000  5595000   521000
  1005000   775000  2200000   582000  1090000   457000  4900000   550000
   570000   220000  4145000   861000  1785000   803000  2565000   674000
   349000  1050000  4425000   406000  4965000  1100000   594000   710000
   692000   650000  2090000   411000  3675000   610000  1675000   530000
  7990000   250000   715000  3500000  2250000   625000  2695000   428000
   695000   930000   455000  1195000  2175000   442000   500000  3395000
   537000   750000  1790000   468000   420000   920000   533000   850000
   751000   388000   490000   494000   525000  2890000   365000  1895000
  5990000  2075000   394000   725000   240000   835000  1725000   330000
   990000   755000  1750000  2083000   271000  1280000   436000  1250000
  2225000   475000   600000  2450000  1625000  1650000   249000   630000
  2599000   385000

In [23]:
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd  # Assuming df_Banglore_1 is a pandas DataFrame

# Assuming df_Banglore_1 is already loaded

# Step 1: Standardize the text
df_Banglore_1['Insurance Validity'] = df_Banglore_1['Insurance Validity'].astype(str).str.strip()

# Step 2: Define the custom order
insurance_order = [
    'Not Available',
    '1',
    '2',
    'Third Party',
    'Third Party insurance',
    'Comprehensive',
    'Zero Dep'
]

# Step 2.1: Inspect unique values in your column
print("Unique values in 'Insurance Validity' before encoding:")
print(df_Banglore_1['Insurance Validity'].unique())

# Step 2.2: Clean or map inconsistent values if necessary
# Example: Mapping numerical strings to the intended categories
mapping = {
    '1.0': '1',
    '2.0': '2',
    '3.0': 'Third Party',
    '4.0': 'Third Party insurance',
    '5.0': 'Comprehensive',
    '6.0': 'Zero Dep',
    '0.0': 'Not Available' # Assuming 0.0 might correspond to 'Not Available'
}
df_Banglore_1['Insurance Validity'] = df_Banglore_1['Insurance Validity'].replace(mapping)

# Re-inspect unique values after potential mapping
print("\nUnique values in 'Insurance Validity' after potential mapping:")
print(df_Banglore_1['Insurance Validity'].unique())

# Step 3: Apply OrdinalEncoder with that order
oe = OrdinalEncoder(categories=[insurance_order])
df_Banglore_1[['Insurance Validity']] = oe.fit_transform(df_Banglore_1[['Insurance Validity']])

# Step 4: Show result
print("\nEncoded 'Insurance Validity' column:")
print(df_Banglore_1[['Insurance Validity']].head())

Unique values in 'Insurance Validity' before encoding:
['4.0' '5.0' '3.0' '6.0' '2.0' '1.0' '0.0']

Unique values in 'Insurance Validity' after potential mapping:
['Third Party insurance' 'Comprehensive' 'Third Party' 'Zero Dep' '2' '1'
 'Not Available']

Encoded 'Insurance Validity' column:
   Insurance Validity
0                 4.0
1                 5.0
2                 5.0
3                 5.0
4                 4.0


In [18]:
df_Banglore_1.head()

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,400000,Hatchback,120000,3,2015,23.10,KA51,Petrol,2015,2015,4.0,5,Manual,5,bangalore,998
1,811000,SUV,32706,2,2018,17.00,KA05,Petrol,2018,2018,5.0,5,Manual,5,bangalore,1497
2,585000,Hatchback,11949,1,2018,23.84,KA03,Petrol,2018,2018,5.0,5,Manual,5,bangalore,1199
3,462000,Sedan,17794,1,2014,19.10,KA53,Petrol,2014,2014,5.0,5,Manual,5,bangalore,1197
4,790000,SUV,60000,1,2015,23.65,KA04,Diesel,2015,2015,4.0,5,Manual,5,bangalore,1248
